# 02 - Degree days and population weighting

Two details that change the answer materially:

1. Degree days accumulate from **daily means**, following VDI 3787. Summing
   hourly deviations instead counts the diurnal cycle as demand.
2. Population weighting requires **summing** a population raster onto the model
   grid, not interpolating it. Interpolation conserves the total while
   scattering people at random.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

from alpinemet.indicators.degree_days import (
    CDD_BASE_TEMPERATURE,
    HDD_BASE_TEMPERATURE,
    degree_days,
    heating_degree_days,
)

## Why daily means matter

In [ ]:
time = pd.date_range("2020-01-01", periods=24 * 30, freq="h")
# Daily mean sitting exactly at the heating base, swinging +/- 8 K through the day.
values = HDD_BASE_TEMPERATURE + 8.0 * np.sin(np.arange(time.size) / 24 * 2 * np.pi)
temperature = xr.DataArray(
    values, dims="time", coords={"time": time}, attrs={"units": "degC"}
)

correct = float(heating_degree_days(temperature))
hourly = float(np.clip(HDD_BASE_TEMPERATURE - values, 0, None).sum() / 24)

print(f"VDI 3787, from daily means : {correct:8.1f} degC d")
print(f"Summing hourly deviations  : {hourly:8.1f} degC d   <- invented demand")

The daily mean never leaves the base temperature, so there is no heating
demand. Accumulating hour by hour invents roughly 76 degree days a month out of
nothing.

## Degree days over a synthetic year

In [ ]:
time = pd.date_range("2020-01-01", "2020-12-31 23:00", freq="h")
day_of_year = time.dayofyear.to_numpy()
hour = time.hour.to_numpy()

seasonal = 10.0 - 12.0 * np.cos(2 * np.pi * (day_of_year - 15) / 365)
diurnal = 4.0 * np.sin(2 * np.pi * (hour - 9) / 24)

lats = np.array([46.5, 47.0, 47.5])
lons = np.array([11.0, 12.0, 13.0])
# Three rows at increasing elevation, hence increasingly cold.
elevation_offset = np.array([0.0, -2.0, -4.0])[:, None] * np.ones(lons.size)

field = (seasonal + diurnal)[:, None, None] + elevation_offset[None, :, :]
temperature = xr.DataArray(
    field,
    dims=("time", "latitude", "longitude"),
    coords={"time": time, "latitude": lats, "longitude": lons},
    attrs={"units": "degC"},
)

result = degree_days(temperature)
print(f"HDD base {HDD_BASE_TEMPERATURE} degC, CDD base {CDD_BASE_TEMPERATURE} degC")
print()
print("annual totals by latitude row (degC d):")
result[["hdd", "cdd"]].mean(dim="longitude").to_pandas()

## Population weighting: summing versus interpolating

A population raster holds counts *per cell*. Moving it to a coarser grid is an
aggregation, not an interpolation.

In [ ]:
from alpinemet.indicators.population import aggregate_population_to_grid

rng = np.random.default_rng(0)
fine_lat = np.linspace(46.0, 48.0, 400)
fine_lon = np.linspace(10.0, 12.0, 400)

people = rng.uniform(0.0, 1.0, (400, 400))   # thin rural scatter everywhere
people[110:130, 110:130] += 500.0            # one compact city

population = xr.DataArray(
    people,
    dims=("latitude", "longitude"),
    coords={"latitude": fine_lat, "longitude": fine_lon},
)

target = xr.DataArray(
    np.zeros((4, 4)),
    dims=("latitude", "longitude"),
    coords={
        "latitude": np.linspace(46.25, 47.75, 4),
        "longitude": np.linspace(10.25, 11.75, 4),
    },
)

correct = aggregate_population_to_grid(population, target)
legacy = aggregate_population_to_grid(population, target, method="nearest_normalised")

city = np.unravel_index(int(np.argmax(correct.values)), correct.shape)

print(f"total, fine grid       : {float(population.sum()):>12,.0f}")
print(f"total, sum aggregation : {float(correct.sum()):>12,.0f}")
print(f"total, nearest+rescale : {float(legacy.sum()):>12,.0f}")
print()
print(f"city cell share, correct: {float(correct.values[city]) / float(correct.sum()):>7.1%}")
print(f"city cell share, legacy : {float(legacy.values[city]) / float(legacy.sum()):>7.1%}")

Both conserve the domain total; only the aggregation puts the people where they
live. The distortion grows with the coarseness of the target grid -- one source
cell in 625 at 2.5 km, one in 96,000 at 31 km -- so it does not affect compared
products equally. `nearest_normalised` exists solely to reproduce earlier output
and carries a warning attribute saying so.

In [ ]:
print(legacy.attrs["warning"])

## Weighting a temperature field

In [ ]:
from alpinemet.indicators.population import population_weighted_mean

grid = temperature.isel(time=0).drop_vars("time")
weights = aggregate_population_to_grid(population, grid)

area_mean = float(temperature.mean())
pop_mean = float(population_weighted_mean(temperature, weights).mean())

print(f"area-averaged mean temperature      : {area_mean:6.2f} degC")
print(f"population-weighted mean temperature: {pop_mean:6.2f} degC")
print()
print("The population sits in the warm valley row, so weighting shifts the")
print("mean upward. Area averages understate what the demand side experiences.")